## Imports y Configuración Inicial

Librerías necesarias para el análisis de opciones y conexión con Interactive Brokers.



In [1]:
# !pip install ib_insync

from ib_insync import IB, util, Option, Stock, Contract
import asyncio
import nest_asyncio
import random
import time

from datetime import datetime
import pandas as pd

import sys
sys.path.insert(0, '../scripts')
from options_hedge import implied_vol_bisect, calculate_implied_q, StraddleForHedge, OptionsHedgeAnalyzer
from straddle import calculate_straddle_greeks_precise
from rates import get_risk_free_rate
from data_loader import load_treasury_curve

---

## 1. Conexión a TWS/IB Gateway

### Objetivo
Establecer una conexión robusta y confiable con TWS (Trader Workstation) o IB Gateway.

### ¿Qué hace esta celda?

1. **Importa librerías necesarias**
   - `ib_insync`: Interfaz Python para Interactive Brokers
   - `nest_asyncio`: Permite event loops anidados en Jupyter
   - `asyncio`: Programación asíncrona

2. **Configura la conexión**
   - HOST: `127.0.0.1` (localhost)
   - PORT: `7497` (Paper Trading) o `7496` (Live Trading)
   - CLIENT_ID: Generado aleatoriamente para evitar conflictos

3. **Función `connect_to_ib_async()`**
   - Verifica si ya existe una conexión activa
   - Reintentos automáticos (máx 3 intentos)
   - Manejo de conexiones corruptas
   - Timeouts configurables (20 segundos)
   - Mensajes de ayuda si falla la conexión

4. **Event Handlers**
   - Detecta desconexiones automáticamente
   - Muestra advertencias cuando se pierde la conexión

### Configuración de MarketDataType

Por defecto, se usa **MarketDataType 4** (Frozen/Delayed):
- GRATUITO - No requiere suscripción a datos en tiempo real
- Datos retrasados de 15 minutos
- Ideal para Paper Trading y desarrollo
- Disponible fuera del horario de mercado

### ¿Qué variables crea?

- `ib`: Objeto global de conexión (se usa en todas las celdas siguientes)

### Resultado esperado



In [11]:
# Instalación de ib_insync si es necesario

# Permitir event loops anidados en Jupyter
nest_asyncio.apply()

# Configuración de conexión
HOST = "127.0.0.1"
PORT = 7497          # Paper trading (7497) o Live (7496)

# Crear instancia global de IB
ib = IB()

# Configurar handlers para manejar desconexiones
def on_disconnected():
    print("⚠ Desconectado de TWS/IB Gateway")

ib.disconnectedEvent += on_disconnected

async def connect_to_ib_async(max_retries=3):
    """
    Conecta a TWS/IB Gateway de forma robusta con reintentos
    
    Parámetros:
    - max_retries: Número máximo de intentos de conexión
    """
    
    for attempt in range(max_retries):
        try:
            # Si ya está conectado, verificar que funcione
            if ib.isConnected():
                try:
                    # Test simple para verificar que la conexión funciona
                    accounts = ib.managedAccounts()
                    print(f"✓ Ya conectado - clientId: {ib.client.clientId}")
                    print(f"  - Server version: {ib.client.serverVersion()}")
                    print(f"  - Managed accounts: {accounts}")
                    return True
                except Exception:
                    # La conexión está corrupta, desconectar
                    print("⚠ Conexión existente corrupta, desconectando...")
                    try:
                        ib.disconnect()
                    except:
                        pass
                    await asyncio.sleep(2)
            
            # Generar CLIENT_ID único basado en timestamp
            # Esto evita conflictos con conexiones previas
            client_id = random.randint(100, 999)
            
            print(f"\nIntento {attempt + 1}/{max_retries}: Conectando con clientId={client_id}...")
            
            # Intentar conectar con timeout más largo
            await ib.connectAsync(HOST, PORT, clientId=client_id, timeout=20, readonly=False)
            
            # Verificar que la conexión funciona
            await asyncio.sleep(1)
            accounts = ib.managedAccounts()
            
            print("✓ Conectado exitosamente")
            print(f"  - Client ID: {client_id}")
            print(f"  - Server version: {ib.client.serverVersion()}")
            print(f"  - Managed accounts: {accounts}")
            
            return True
            
        except Exception as e:
            error_msg = str(e)
            print(f"✗ Error en intento {attempt + 1}: {error_msg}")
            
            # Limpiar conexión fallida
            try:
                if ib.isConnected():
                    ib.disconnect()
            except:
                pass
            
            # Si es el último intento, mostrar mensaje de ayuda
            if attempt == max_retries - 1:
                print("\n" + "="*80)
                print("ERROR: No se pudo conectar a TWS/IB Gateway")
                print("="*80)
                print("\nVerifica lo siguiente:")
                print("  1. TWS o IB Gateway está ejecutándose")
                print(f"  2. Está configurado para aceptar conexiones API en el puerto {PORT}")
                print("  3. La configuración API permite conexiones desde 127.0.0.1")
                print("  4. No hay otras conexiones activas bloqueando el acceso")
                print("\nPasos para configurar TWS:")
                print("  - File -> Global Configuration -> API -> Settings")
                print("  - Marcar 'Enable ActiveX and Socket Clients'")
                print(f"  - Socket port: {PORT}")
                print("  - Trusted IP addresses: 127.0.0.1")
                print("="*80)
                return False
            
            # Esperar antes del siguiente intento
            wait_time = (attempt + 1) * 2
            print(f"  Esperando {wait_time} segundos antes del siguiente intento...")
            await asyncio.sleep(wait_time)
    
    return False

# Ejecutar la conexión
print("Iniciando conexión a TWS/IB Gateway...")
print("="*80)
connection_successful = await connect_to_ib_async(max_retries=3)

if connection_successful:
    print("\n✓ Listo para operar")
else:
    print("\n✗ No se pudo establecer conexión. Revisa los mensajes anteriores.")

Iniciando conexión a TWS/IB Gateway...

Intento 1/3: Conectando con clientId=619...
✓ Conectado exitosamente
  - Client ID: 619
  - Server version: 176
  - Managed accounts: ['DUP064233']

✓ Listo para operar


---

## 2. Obtener Cadena de Opciones

### Objetivo
Descargar la cadena completa de opciones para SPY con un vencimiento específico.

### ¿Qué hace esta celda?

1. **Función `verify_connection()`**
  - Verifica que la conexión con IBKR esté activa
  - Lanza error si la conexión está corrupta o inactiva

2. **Función `get_option_chain_async()`**
  - **Parámetros:**
    - `symbol`: Símbolo del subyacente (default: "SPY")
    - `skip_expiries`: Número de vencimientos a saltar (1 = siguiente vencimiento, 2 = el siguiente, etc.)
  
  - **Proceso:**
    1. Cualifica el contrato del subyacente (SPY)
    2. Solicita parámetros de opciones disponibles
    3. Selecciona la cadena estándar (exchange=SMART)
    4. Filtra vencimientos futuros
    5. Selecciona vencimiento automáticamente
    6. Descarga todos los contratos (CALLs y PUTs)
    7. Construye DataFrame con la cadena completa

3. **Estructura de datos retornada**
  - Tabla pivoteada: cada fila = un strike
  - Columnas:
    - `strike`: Precio de ejercicio
    - `call_localSymbol`: Símbolo local del CALL
    - `call_conId`: ID del contrato CALL
    - `put_localSymbol`: Símbolo local del PUT
    - `put_conId`: ID del contrato PUT

### ¿Qué variables crea?

- `df_option_chain`: DataFrame con la cadena completa de opciones
- `expiry_date`: Fecha de vencimiento seleccionada (formato: YYYYMMDD)

### Parámetros configurables



In [ ]:
async def verify_connection():
    """Verifica que la conexión esté activa y funcionando"""
    if not ib.isConnected():
        raise RuntimeError("No hay conexión activa. Ejecuta primero la celda de conexión.")
    
    try:
        # Test simple para verificar que la conexión funciona
        ib.managedAccounts()
    except Exception as e:
        raise RuntimeError(f"Conexión corrupta: {e}. Ejecuta nuevamente la celda de conexión.")

async def get_option_chain_async(symbol="SPY", skip_expiries=1):
    """
    Obtiene la cadena de opciones de forma robusta
    
    Parámetros:
    - symbol: Símbolo del subyacente (default: SPY)
    - skip_expiries: Número de vencimientos a saltar desde hoy (1 = siguiente vencimiento)
    
    Returns:
    - Tuple: (df_chain, expiry) - DataFrame con la cadena y fecha de vencimiento
    """
    
    print("="*80)
    print(f"OBTENIENDO CADENA DE OPCIONES: {symbol}")
    print("="*80 + "\n")
    
    # Verificar conexión
    await verify_connection()
    print(f"✓ Conexión activa (clientId={ib.client.clientId})\n")
    
    try:
        # -----------------------------
        # 1) Definir el subyacente
        # -----------------------------
        print(f"1. Cualificando contrato del subyacente {symbol}...")
        underlying = Stock(symbol, "SMART", "USD")
        await ib.qualifyContractsAsync(underlying)
        print(f"   ✓ {underlying.symbol} - ConId: {underlying.conId}\n")
        
        # -----------------------------
        # 2) Obtener parámetros de opciones
        # -----------------------------
        print("2. Solicitando parámetros de cadenas de opciones...")
        chains = await ib.reqSecDefOptParamsAsync(symbol, "", "STK", underlying.conId)
        
        if not chains:
            raise RuntimeError(f"No se encontraron cadenas de opciones para {symbol}")
        
        # Elegir cadena estándar (no FLEX)
        chain = next((c for c in chains if c.exchange == "SMART" and c.tradingClass == symbol), None)
        
        if not chain:
            raise RuntimeError(f"No se encontró cadena estándar para {symbol}")
        
        print(f"   ✓ Cadena: exchange={chain.exchange}, tradingClass={chain.tradingClass}, multiplier={chain.multiplier}\n")
        
        # -----------------------------
        # 3) Seleccionar vencimiento
        # -----------------------------
        expirations = sorted(chain.expirations)
        today = datetime.now().strftime("%Y%m%d")
        
        # Filtrar vencimientos futuros
        future_expiries = [e for e in expirations if e > today]
        
        if not future_expiries:
            raise RuntimeError("No hay vencimientos futuros disponibles")
        
        # Seleccionar vencimiento (skip_expiries para evitar 0DTE si es necesario)
        expiry_index = min(skip_expiries, len(future_expiries) - 1)
        expiry = future_expiries[expiry_index]
        
        print(f"3. Vencimiento seleccionado: {expiry}")
        print(f"   Vencimientos disponibles: {len(future_expiries)}")
        n = 30 # definir cuántos mostrar vencimientos
        print(f"   Primeros {n}: {future_expiries[:n]}\n")
        
        # -----------------------------
        # 4) Obtener contratos de opciones
        # -----------------------------
        print("4. Solicitando contratos de opciones a IBKR...")
        print("   (esto puede tomar unos segundos...)\n")
        
        tmpl = Contract(
            secType="OPT",
            symbol=symbol,
            currency="USD",
            exchange="SMART",
            lastTradeDateOrContractMonth=expiry,
            multiplier=str(chain.multiplier),
            tradingClass=chain.tradingClass,
        )
        
        details = await ib.reqContractDetailsAsync(tmpl)
        
        if not details:
            raise RuntimeError(
                f"No se recibieron contratos para {symbol} vencimiento {expiry}. "
                "Verifica permisos de market data o intenta otro vencimiento."
            )
        
        print(f"   ✓ Recibidos {len(details)} contratos\n")
        
        # -----------------------------
        # 5) Construir DataFrame
        # -----------------------------
        print("5. Procesando contratos...")
        
        rows = []
        for d in details:
            c = d.contract
            if c.symbol != symbol or c.secType != "OPT":
                continue
            rows.append({
                "expiry": c.lastTradeDateOrContractMonth,
                "right": c.right,
                "strike": c.strike,
                "conId": c.conId,
                "localSymbol": c.localSymbol,
                "exchange": c.exchange,
                "tradingClass": getattr(c, "tradingClass", None),
            })
        
        df = pd.DataFrame(rows)
        
        # Limpiar y filtrar
        df = df[df["expiry"] == expiry]
        df = df[df["right"].isin(["C", "P"])]
        df = df.dropna(subset=["strike", "conId"])
        df["strike"] = df["strike"].astype(float)
        
        print(f"   ✓ {len(df)} contratos procesados\n")
        
        # -----------------------------
        # 6) Crear tabla Option Chain
        # -----------------------------
        print("6. Creando tabla de option chain...")
        
        df_calls = (
            df[df["right"] == "C"][["strike", "localSymbol", "conId"]]
            .rename(columns={"localSymbol": "call_localSymbol", "conId": "call_conId"})
        )
        df_puts = (
            df[df["right"] == "P"][["strike", "localSymbol", "conId"]]
            .rename(columns={"localSymbol": "put_localSymbol", "conId": "put_conId"})
        )
        
        df_chain = (
            df_calls.merge(df_puts, on="strike", how="outer")
            .sort_values("strike")
            .reset_index(drop=True)
        )
        
        # Mostrar resumen
        print(f"\n{'='*80}")
        print(f"OPTION CHAIN {symbol} - Expiry: {expiry}")
        print(f"{'='*80}")
        print(f"Total strikes: {len(df_chain)}")
        print(f"Rango: ${df_chain['strike'].min():.0f} - ${df_chain['strike'].max():.0f}")
        print(f"\nPrimeros 10 strikes:")
        print(df_chain.head(10)[['strike', 'call_localSymbol', 'put_localSymbol']].to_string(index=False))
        print(f"{'='*80}\n")
        
        print("✓ Cadena de opciones obtenida exitosamente\n")
        
        return df_chain, expiry
        
    except Exception as e:
        print(f"\n✗ ERROR: {e}\n")
        raise

# Ejecutar la función
try:
    df_option_chain, expiry_date = await get_option_chain_async("SPY", skip_expiries=2) # INIDCAR LOS VENCIMIENTOS A SALTAR DESDE HOY
    print(f"Variable 'df_option_chain' creada con {len(df_option_chain)} strikes")
    print(f"Variable 'expiry_date' = {expiry_date}")
except Exception as e:
    print(f"No se pudo obtener la cadena de opciones: {e}")

OBTENIENDO CADENA DE OPCIONES: SPY

✓ Conexión activa (clientId=619)

1. Cualificando contrato del subyacente SPY...
   ✓ SPY - ConId: 756733

2. Solicitando parámetros de cadenas de opciones...
   ✓ Cadena: exchange=SMART, tradingClass=SPY, multiplier=100

3. Vencimiento seleccionado: 20280121
   Vencimientos disponibles: 32
   Primeros 30: ['20260114', '20260115', '20260116', '20260120', '20260121', '20260122', '20260123', '20260126', '20260127', '20260130', '20260206', '20260213', '20260220', '20260227', '20260320', '20260331', '20260417', '20260430', '20260529', '20260618', '20260630', '20260918', '20260930', '20261218', '20261231', '20270115', '20270319', '20270617', '20271217', '20280121']

4. Solicitando contratos de opciones a IBKR...
   (esto puede tomar unos segundos...)

   ✓ Recibidos 458 contratos

5. Procesando contratos...
   ✓ 458 contratos procesados

6. Creando tabla de option chain...

OPTION CHAIN SPY - Expiry: 20280121
Total strikes: 229
Rango: $50 - $1340

Primero

---

## 3. Obtener Precio Actual de SPY

###  Objetivo
Obtener el precio actual del subyacente (SPY) para identificar el strike ATM (At The Money).

###  ¿Qué hace esta celda?

Esta celda implementa un **sistema de fallback robusto en 3 niveles** para asegurar que siempre obtengamos un precio de referencia:

#### **MÉTODO 1: IBKR (Prioridad Alta)**
- Usa la conexión activa con IBKR
- Solicita datos de mercado retrasados (MarketDataType 4)
- Prioridad de precios:
  1. `Last` (último precio negociado)
  2. `Close` (precio de cierre)
  3. `Bid` (mejor oferta de compra)
-  **Gratis** - No requiere suscripción a datos en tiempo real

#### **MÉTODO 2: Yahoo Finance (Fallback)**
- Si IBKR no devuelve precio válido
- Descarga datos usando la librería `yfinance`
- Instala automáticamente `yfinance` si no está disponible
- Prioridad de precios:
  1. `regularMarketPrice` (precio de mercado regular)
  2. `currentPrice` (precio actual)
  3. `previousClose` (cierre anterior)
  4. `Close` del historial
-  **Gratis** - No requiere API key

#### **MÉTODO 3: Precio Manual (Última Opción)**
- Si ambos métodos anteriores fallan
- Usa precio de seguridad hardcodeado (~$595.00)
-  **Advertencia clara al usuario** para verificar conexión
- Permite que el notebook continúe funcionando

###  ¿Qué variables crea?

- `spy_price`: Precio actual de SPY (float)

###  Resultado esperado

**Caso exitoso (IBKR):**
```
 ÉXITO - Precio obtenido de IBKR
================================================================================
PRECIO SPY: $689.51
Fuente: IBKR (Last)
================================================================================
```

**Caso exitoso (Yahoo):**
```
 ÉXITO - Precio obtenido de Yahoo Finance
================================================================================
PRECIO SPY: $689.51
Fuente: Yahoo Finance (Regular Market Price)
================================================================================
```

**Caso fallback:**
```
 ADVERTENCIA: Usando precio manual de seguridad
================================================================================
PRECIO SPY: $595.00
Fuente: Precio Manual de Seguridad
================================================================================

  IMPORTANTE: Este es un precio de respaldo.
   Verifica la conexión IBKR o la disponibilidad de Yahoo Finance
```

###  Tiempo de ejecución

- IBKR: **1-3 segundos**
- Yahoo Finance: **2-5 segundos**
- Manual: **Instantáneo**

###  Notas importantes

-  **Triple capa de seguridad** - Siempre obtendrás un precio
-  **No bloquea el flujo** - El notebook puede continuar
-  **Instalación automática** de dependencias (yfinance)
-  **Mensajes claros** indicando la fuente del precio
-  Si ves el precio manual, verifica tu conexión

###  ¿Cuándo actualizar el precio manual?

Si el precio de SPY cambia significativamente (~>10%), actualiza esta línea en el código:

```python
safety_price = 595.00  # <-- Actualizar aquí
```

---

** Ejecuta esta celda después de obtener la cadena de opciones**

In [4]:
async def get_spy_price_async():
    """
    Obtiene el precio actual de SPY con fallback robusto:
    1. Intenta obtenerlo de IBKR (datos delayed/frozen)
    2. Si falla, descarga de Yahoo Finance
    3. Si falla, usa precio manual de seguridad
    
    Returns:
    - float: Precio actual de SPY
    """
    
    print("="*80)
    print("OBTENIENDO PRECIO DE SPY (Con Fallback Robusto)")
    print("="*80 + "\n")
    
    # =========================================================================
    # MÉTODO 1: Obtener de IBKR
    # =========================================================================
    print(" MÉTODO 1: Intentando obtener precio desde IBKR...")
    print("-" * 80)
    
    if ib.isConnected():
        try:
            # Configurar datos retrasados/congelados (MarketDataType 4)
            ib.reqMarketDataType(3)
            print("✓ Conexión activa - MarketDataType(4) configurado\n")
            
            # Crear contrato de SPY
            print("1. Cualificando contrato SPY...")
            spy = Stock("SPY", "SMART", "USD")
            await ib.qualifyContractsAsync(spy)
            print(f"   ✓ Contrato cualificado - ConId: {spy.conId}\n")
            
            # Solicitar datos de mercado (snapshot)
            print("2. Solicitando precio de mercado...")
            spy_ticker = ib.reqMktData(spy, "", snapshot=False, regulatorySnapshot=False)
            print(spy_ticker)
            # Esperar a recibir datos (máximo 3 segundos)
            spy_price = None
            for i in range(15):
                await asyncio.sleep(1)
                
                # Intentar obtener precio en orden de prioridad
                if spy_ticker.last and spy_ticker.last > 0:
                    spy_price = spy_ticker.last
                    price_type = "Last"
                    break
                elif spy_ticker.close and spy_ticker.close > 0:
                    spy_price = spy_ticker.close
                    price_type = "Close"
                    break
                elif spy_ticker.bid and spy_ticker.bid > 0:
                    spy_price = spy_ticker.bid
                    price_type = "Bid"
                    break
            
            # Cancelar suscripción
            ib.cancelMktData(spy)
            
            # Verificar resultado
            if spy_price and spy_price > 0:
                print(f" ÉXITO - Precio obtenido de IBKR")
                print("="*80)
                print(f"PRECIO SPY: ${spy_price:.2f}")
                print(f"Fuente: IBKR ({price_type})")
                print("="*80 + "\n")
                return spy_price
            else:
                print(" IBKR no devolvió precio válido\n")
                
        except Exception as e:
            print(f" Error al obtener precio de IBKR: {e}\n")
    else:
        print(" No hay conexión activa con IBKR\n")
    
    # =========================================================================
    # MÉTODO 2: Obtener de Yahoo Finance
    # =========================================================================
    print(" MÉTODO 2: Intentando obtener precio desde Yahoo Finance...")
    print("-" * 80)
    
    try:
        # Intentar importar yfinance
        try:
            import yfinance as yf
        except ImportError:
            print(" Librería 'yfinance' no instalada.")
            print("  Instalando automáticamente...")
            import subprocess
            import sys
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "yfinance"])
            import yfinance as yf
            print("✓ yfinance instalado correctamente\n")
        
        print("1. Descargando datos de Yahoo Finance...")
        spy_ticker = yf.Ticker("SPY")
        
        # Obtener datos de diferentes fuentes
        spy_info = spy_ticker.info
        spy_history = spy_ticker.history(period="1d")
        
        spy_price = None
        price_type = None
        
        # Prioridad: regularMarketPrice > currentPrice > previousClose > Close del history
        if 'regularMarketPrice' in spy_info and spy_info['regularMarketPrice']:
            spy_price = spy_info['regularMarketPrice']
            price_type = "Regular Market Price"
        elif 'currentPrice' in spy_info and spy_info['currentPrice']:
            spy_price = spy_info['currentPrice']
            price_type = "Current Price"
        elif 'previousClose' in spy_info and spy_info['previousClose']:
            spy_price = spy_info['previousClose']
            price_type = "Previous Close"
        elif not spy_history.empty:
            spy_price = spy_history['Close'].iloc[-1]
            price_type = "Last Close (History)"
        
        if spy_price and spy_price > 0:
            print(f" ÉXITO - Precio obtenido de Yahoo Finance")
            print("="*80)
            print(f"PRECIO SPY: ${spy_price:.2f}")
            print(f"Fuente: Yahoo Finance ({price_type})")
            print("="*80 + "\n")
            return float(spy_price)
        else:
            print(" Yahoo Finance no devolvió precio válido\n")
            
    except Exception as e:
        print(f" Error al obtener precio de Yahoo Finance: {e}\n")
    
    # =========================================================================
    # MÉTODO 3: Precio Manual de Seguridad
    # =========================================================================
    print(" MÉTODO 3: Usando precio manual de seguridad...")
    print("-" * 80)
    
    # Precio de seguridad (actualizar manualmente si es necesario)
    # Este es un precio aproximado del SPY a finales de diciembre 2024
    safety_price = 595.00
    
    print(f" ADVERTENCIA: Usando precio manual de seguridad")
    print("="*80)
    print(f"PRECIO SPY: ${safety_price:.2f}")
    print(f"Fuente: Precio Manual de Seguridad")
    print("="*80)
    print("\n  IMPORTANTE: Este es un precio de respaldo.")
    print("   Verifica la conexión IBKR o la disponibilidad de Yahoo Finance")
    print("   para obtener precios actualizados en tiempo real.\n")
    
    return safety_price

# Ejecutar la función
spy_price = await get_spy_price_async()

print(f"✓ Variable 'spy_price' = ${spy_price:.2f}")

OBTENIENDO PRECIO DE SPY (Con Fallback Robusto)

 MÉTODO 1: Intentando obtener precio desde IBKR...
--------------------------------------------------------------------------------
✓ Conexión activa - MarketDataType(4) configurado

1. Cualificando contrato SPY...
   ✓ Contrato cualificado - ConId: 756733

2. Solicitando precio de mercado...
Ticker(contract=Stock(conId=756733, symbol='SPY', exchange='SMART', primaryExchange='ARCA', currency='USD', localSymbol='SPY', tradingClass='SPY'))
 IBKR no devolvió precio válido

 MÉTODO 2: Intentando obtener precio desde Yahoo Finance...
--------------------------------------------------------------------------------
1. Descargando datos de Yahoo Finance...
 ÉXITO - Precio obtenido de Yahoo Finance
PRECIO SPY: $693.96
Fuente: Yahoo Finance (Regular Market Price)

✓ Variable 'spy_price' = $693.96


---

## 4. Obtener Precios Bid/Ask de Opciones

###  Objetivo
Obtener precios de mercado (bid/ask) para las opciones cercanas al precio ATM de SPY.

###  ¿Qué hace esta celda?

#### **Función `get_option_prices_async()`**

**Parámetros:**
- `df_chain`: DataFrame con la cadena de opciones (del paso anterior)
- `expiry_date`: Fecha de vencimiento (del paso anterior)
- `center_strike`: Strike central específico (opcional)
- `max_strikes`: Número máximo de strikes a procesar (default: 20)
- `reference_price`: Precio de referencia de SPY (del paso anterior)

**Proceso:**

| Paso | Descripción | Detalles |
|------|-------------|----------|
| **1. Configuración de datos** | Usa MarketDataType(4) - Frozen/Delayed |  **GRATUITO** - No requiere suscripción a datos en tiempo real. Ideal para Paper Trading y desarrollo |
| **2. Selección de strikes** | Filtra strikes cercanos al ATM | Usa el precio de referencia de SPY. Encuentra el strike más cercano (ATM). Selecciona `max_strikes` alrededor del ATM. Ejemplo: Si ATM=$690 y max_strikes=20 → strikes de $680 a $700 |
| **3. Creación de contratos** | Prepara objetos Option | Crea objetos `Option` para cada CALL y PUT. Cualifica contratos con IBKR. Prepara para solicitud masiva de datos |
| **4. Solicitud de precios** | **MODO SNAPSHOT** -  Solicitud paralela masiva |  **Solicitud paralela masiva** - Todos los contratos a la vez. `snapshot=True` - Solicita dato puntual (no streaming). Espera inteligente: 2-4 segundos con verificación del 90%.  **Mucho más rápido** que solicitudes secuenciales |
| **5. Procesamiento** | Extrae y organiza datos | Extrae: bid, ask, last, volume. Prioriza: Last → Close (para datos delayed). Guarda en DataFrame con columnas separadas para CALL y PUT. Cancela suscripciones para liberar recursos |
| **6. Visualización** | Muestra resultados formateados | Tabla formateada con todos los strikes. Precios bid/ask/last para CALLs y PUTs. Estadísticas de cobertura |

###  ¿Qué variables crea?

- `df_with_prices`: DataFrame con strikes y precios completos

**Columnas del DataFrame:**
- `strike`: Precio de ejercicio
- `call_localSymbol`: Símbolo local del CALL
- `call_conId`: ID del contrato CALL
- `call_bid`: Precio bid del CALL
- `call_ask`: Precio ask del CALL
- `call_last`: Último precio del CALL
- `call_volume`: Volumen del CALL
- `put_localSymbol`: Símbolo local del PUT
- `put_conId`: ID del contrato PUT
- `put_bid`: Precio bid del PUT
- `put_ask`: Precio ask del PUT
- `put_last`: Último precio del PUT
- `put_volume`: Volumen del PUT

###  Resultado esperado

```
====================================================================================================
PRECIOS DE OPCIONES SPY - Expiry: 20251231 (Snapshot Mode)
====================================================================================================

 strike  call_bid  call_ask  call_last  put_bid  put_ask  put_last
  680.0     10.20     11.23      11.02     0.51     0.52      0.52
  681.0      9.60     10.00       9.80     0.58     0.59      0.57
  ...
  690.0      2.59      2.60       2.59     2.28     2.30      2.28  ← ATM
  ...
  700.0      0.09      0.10       0.09     8.84    10.20      8.56

====================================================================================================
Strikes: 21 | Contratos con precios: 42/42
====================================================================================================
```

###  Ventajas del Modo Snapshot

-  **Solicitud paralela masiva** - Todos los contratos simultáneamente
-  **No requiere streaming** - Solo un dato puntual
-  **Gratis con datos delayed** - No requiere suscripción
-  **Espera inteligente** - Verifica que llegue el 90% de datos
-  **Limpieza automática** - Cancela suscripciones al terminar

###  Parámetros configurables

```python
# Obtener más strikes (ejemplo: 40 strikes)
df_with_prices = await get_option_prices_async(
    df_option_chain, 
    expiry_date, 
    max_strikes=40,
    reference_price=spy_price
)

# Forzar un strike central específico
df_with_prices = await get_option_prices_async(
    df_option_chain, 
    expiry_date, 
    center_strike=690.0,
    reference_price=spy_price
)
```

###  Uso para Long Straddle

Con estos datos puedes:
1. Identificar el strike ATM exacto
2. Ver el costo del CALL ATM (call_ask)
3. Ver el costo del PUT ATM (put_ask)
4. **Costo total = call_ask + put_ask**
5. Calcular puntos de equilibrio (strike ± costo total)

###  Notas importantes

-  Usa precios **delayed/frozen** - GRATUITOS
-  Modo snapshot - Mucho más rápido y eficiente
-  Filtra strikes cercanos al ATM automáticamente
-  Durante horario de mercado: datos con 15 min de retraso
-  Fuera de horario: datos "frozen" del último cierre

---

** Ejecuta esta celda después de obtener el precio de SPY**

In [5]:
async def get_option_prices_async(df_chain, expiry_date, center_strike=None, max_strikes=20, reference_price=None):
    """
    Obtiene precios bid/ask de forma robusta usando Snapshot y MarketDataType 4 (Frozen).
    
    Parámetros:
    - df_chain: DataFrame con la cadena de opciones
    - expiry_date: Fecha de vencimiento
    - center_strike: Strike central específico (opcional)
    - max_strikes: Número máximo de strikes a procesar
    - reference_price: Precio de referencia del subyacente (si ya se obtuvo previamente)
    """
    
    print("="*80)
    print(f"OBTENIENDO PRECIOS DE OPCIONES - Expiry: {expiry_date}")
    print("="*80 + "\n")
    
    # Configurar datos congelados/retrasados (Frozen es mejor para pruebas off-hours o delayed)
    ib.reqMarketDataType(4) 
    print("✓ Configurado: MarketDataType(4) - Frozen/Delayed (Más robusto para Paper Trading)\n")
    
    try:
        # -----------------------------
        # 1) Obtener precio de referencia (SPY)
        # -----------------------------
        spy_price = reference_price  # Usar el precio proporcionado como parámetro
        
        if spy_price:
            print(f"1. Usando precio de referencia proporcionado: ${spy_price:.2f}")
            print("   (Obtenido previamente de Yahoo Finance o IBKR)\n")
        else:
            print("1. Obteniendo precio de referencia del subyacente (SPY)...")
            
            try:
                spy = Stock("SPY", "SMART", "USD")
                await ib.qualifyContractsAsync(spy)
                
                # Usamos snapshot=True para asegurar un dato puntual
                spy_ticker = ib.reqMktData(spy, "", True, False)
                
                # Esperar hasta obtener un dato válido (máx 2 seg)
                for _ in range(10):
                    await asyncio.sleep(0.2)
                    # Prioridad: Last -> Close -> MarketPrice
                    if spy_ticker.last and spy_ticker.last > 0:
                        spy_price = spy_ticker.last
                        break
                    elif spy_ticker.close and spy_ticker.close > 0:
                        spy_price = spy_ticker.close
                        break
                
                # Si sigue fallando, intentar marketPrice() genérico de ib_insync
                if not spy_price:
                    spy_price = spy_ticker.marketPrice()
                    
                if spy_price and spy_price > 0:
                    print(f"   ✓ Precio SPY detectado: ${spy_price:.2f}\n")
                else:
                    print("    ADVERTENCIA: No se pudo obtener precio SPY. Usando strike central por defecto.\n")
                    
            except Exception as e:
                print(f"   Error obteniendo SPY: {e}\n")
        
        # -----------------------------
        # 2) Seleccionar strikes
        # -----------------------------
        print("2. Seleccionando strikes...")
        
        # Determinar strike central
        if center_strike is None:
            if spy_price and spy_price > 0:
                temp_df = df_chain.copy()
                temp_df['distance'] = abs(temp_df['strike'] - spy_price)
                center_strike = temp_df.loc[temp_df['distance'].idxmin(), 'strike']
            else:
                # Fallback si no hay precio de SPY: usar la mediana de la cadena
                center_strike = df_chain['strike'].median()
                print(f"   (Usando mediana de la cadena como referencia: {center_strike})")
        
        # Filtrar strikes alrededor del centro
        strike_range = max_strikes // 2
        df_filtered = df_chain.copy().sort_values('strike').reset_index(drop=True)
        
        df_filtered['distance'] = abs(df_filtered['strike'] - center_strike)
        center_idx = df_filtered['distance'].idxmin()
        
        start_idx = max(0, center_idx - strike_range)
        end_idx = min(len(df_filtered), center_idx + strike_range + 1)
        df_filtered = df_filtered.iloc[start_idx:end_idx].copy()
        df_filtered = df_filtered.drop('distance', axis=1).reset_index(drop=True)
        
        print(f"   ✓ Strike central: ${center_strike:.2f}")
        print(f"   ✓ Rango: ${df_filtered['strike'].min():.0f} - ${df_filtered['strike'].max():.0f}")
        print(f"   ✓ {len(df_filtered)} strikes seleccionados\n")
        
        # -----------------------------
        # 3) Crear y Cualificar contratos
        # -----------------------------
        print("3. Creando y cualificando contratos...")
        
        contracts = []
        contract_info = []  # Lista en lugar de diccionario (evita problemas de hash)
        
        for idx, row in df_filtered.iterrows():
            strike = row['strike']
            
            # CALL
            if pd.notna(row.get('call_conId')):
                call = Option('SPY', expiry_date, strike, 'C', 'SMART', tradingClass='SPY')
                contracts.append(call)
                contract_info.append(('call', idx))  # Guardar tipo y índice
            
            # PUT
            if pd.notna(row.get('put_conId')):
                put = Option('SPY', expiry_date, strike, 'P', 'SMART', tradingClass='SPY')
                contracts.append(put)
                contract_info.append(('put', idx))  # Guardar tipo y índice
        
        qualified = await ib.qualifyContractsAsync(*contracts)
        print(f"   ✓ {len(qualified)} contratos listos para solicitar datos\n")
        
        # -----------------------------
        # 4) Obtener precios (SNAPSHOT / PARALELO)
        # -----------------------------
        print("4. Solicitando Snapshots de mercado (Paralelo)...")
        
        # Inicializar columnas en el DF
        cols_to_init = ['call_bid', 'call_ask', 'call_last', 'call_volume', 
                       'put_bid', 'put_ask', 'put_last', 'put_volume']
        for col in cols_to_init:
            df_filtered[col] = None

        # SOLICITUD MASIVA (Mucho más rápido)
        # Usamos snapshot=True. Esto es clave para cuentas sin datos live.
        tickers_list = []
        for contract in qualified:
            # snapshot=True, regulatorySnapshot=False
            t = ib.reqMktData(contract, "", True, False)
            tickers_list.append(t)
        
        print(f"    Esperando datos (2-4 segundos)...")
        # Esperamos un tiempo prudencial para que lleguen los snapshots
        await asyncio.sleep(20) 
        
        # Opcional: Espera activa inteligente (esperar a que la mayoría tenga datos)
        for _ in range(20):  # Máx 4 segundos extra
            filled = sum(1 for t in tickers_list if t.last or t.bid or t.ask or t.close)
            if filled >= len(qualified) * 0.9:  # Si tenemos el 90% de los datos
                break
            await asyncio.sleep(0.2)

        # -----------------------------
        # 5) Procesar resultados
        # -----------------------------
        valid_count = 0

        # Procesamos los resultados usando índices
        # contract_info, qualified y tickers_list están alineados por índice
        for i, (qualified_contract, ticker) in enumerate(zip(qualified, tickers_list)):

            # Obtener información usando el índice
            if i >= len(contract_info):
                continue
                
            opt_type, df_idx = contract_info[i]

            # Lógica de precios (Priorizamos Last, luego Close para Delayed)
            bid = ticker.bid if (ticker.bid and ticker.bid > 0) else None
            ask = ticker.ask if (ticker.ask and ticker.ask > 0) else None
            last = ticker.last if (ticker.last and ticker.last > 0) else ticker.close
            volume = ticker.volume

            # Asignar al DataFrame
            prefix = 'call' if opt_type == 'call' else 'put'

            df_filtered.at[df_idx, f'{prefix}_bid'] = bid
            df_filtered.at[df_idx, f'{prefix}_ask'] = ask
            df_filtered.at[df_idx, f'{prefix}_last'] = last
            df_filtered.at[df_idx, f'{prefix}_volume'] = volume

            if bid or ask or last:
                valid_count += 1

            # Limpiar suscripción (buena práctica)
            ib.cancelMktData(qualified_contract)

        # -----------------------------
        # 6) Mostrar resultados
        # -----------------------------
        print(f"\n{'='*100}")
        print(f"PRECIOS DE OPCIONES SPY - Expiry: {expiry_date} (Snapshot Mode)")
        print(f"{'='*100}\n")
        
        display_cols = ['strike', 'call_bid', 'call_ask', 'call_last', 'put_bid', 'put_ask', 'put_last']
        
        # Formatear para imprimir bonito (rellenar NaN con guiones)
        df_print = df_filtered[display_cols].fillna('-')
        print(df_print.to_string(index=False))
        
        print(f"\n{'='*100}")
        print(f"Strikes: {len(df_filtered)} | Contratos con precios: {valid_count}/{len(qualified)}")
        print(f"{'='*100}\n")
        
        return df_filtered
        
    except Exception as e:
        print(f"\n✗ ERROR CRÍTICO: {e}")
        import traceback
        traceback.print_exc()
        return df_chain  # Devolver original en caso de error

# ---------------------------------------------------------
# Bloque de ejecución principal
# ---------------------------------------------------------
# Ejecutar la función pasando el precio obtenido de Yahoo Finance
df_with_prices = await get_option_prices_async(df_option_chain, expiry_date, reference_price=spy_price)

OBTENIENDO PRECIOS DE OPCIONES - Expiry: 20260116

✓ Configurado: MarketDataType(4) - Frozen/Delayed (Más robusto para Paper Trading)

1. Usando precio de referencia proporcionado: $693.96
   (Obtenido previamente de Yahoo Finance o IBKR)

2. Seleccionando strikes...
   ✓ Strike central: $694.00
   ✓ Rango: $684 - $704
   ✓ 21 strikes seleccionados

3. Creando y cualificando contratos...
   ✓ 42 contratos listos para solicitar datos

4. Solicitando Snapshots de mercado (Paralelo)...
    Esperando datos (2-4 segundos)...

PRECIOS DE OPCIONES SPY - Expiry: 20260116 (Snapshot Mode)

 strike  call_bid  call_ask  call_last  put_bid  put_ask  put_last
  684.0     11.27     11.49      11.95     0.65     0.66      0.70
  685.0     10.36     10.58      10.50     0.74     0.76      0.73
  686.0      9.46      9.70       9.90     0.85     0.87      1.00
  687.0      8.59      8.82       8.69     0.98     1.00      0.99
  688.0      7.76      7.97       7.80     1.13     1.15      1.22
  689.0    

C:\Users\diego\AppData\Local\Temp\ipykernel_23060\596891257.py:196: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_print = df_filtered[display_cols].fillna('-')


---

## 5. Envío de Órdenes - Long Straddle

### Objetivo
Enviar órdenes para ejecutar un Long Straddle ATM usando las variables calculadas previamente (`df_with_prices`, `expiry_date`, `df_option_chain`).

### Modos de Envío Disponibles

| Modo | Descripción | Ventajas | Desventajas |
|------|-------------|----------|-------------|
| **COMBO (BAG)** | Una sola orden para ambas patas | Ejecución simultánea garantizada, menos comisiones | Puede tener spread más amplio si hay poco volumen en el combo |
| **INDIVIDUAL** | Órdenes separadas CALL + PUT | Mejor precio en cada pata individual | Riesgo de legging (ejecución parcial), más comisiones |

### Tipos de Orden

| Tipo | Descripción | Uso Recomendado |
|------|-------------|-----------------|
| **MarketOrder (MKT)** | Ejecución inmediata al mejor precio disponible | Alta liquidez, urgencia, spreads estrechos |
| **LimitOrder (LMT)** | Ejecución solo si precio ≤ límite especificado | Control de precio, spreads amplios |

### Variables Reutilizadas

Esta celda usa las variables calculadas en los pasos anteriores:
- `ib`: Conexión activa con IB Gateway (NO se reconecta)
- `df_option_chain`: DataFrame con conIds de todos los strikes
- `df_with_prices`: DataFrame con precios bid/ask actuales
- `expiry_date`: Fecha de vencimiento seleccionada
- `spy_price`: Precio actual del subyacente

### Resultado Esperado

```
================================================================================
ENVÍO DE ÓRDENES - LONG STRADDLE SPY
================================================================================

Strike ATM seleccionado: $694.00
  CALL conId: 123456789
  PUT conId:  987654321

Precios de mercado:
  CALL: bid=$2.80 / ask=$2.82
  PUT:  bid=$2.66 / ask=$2.68

================================================================================
RESUMEN DE ORDEN - LONG STRADDLE
================================================================================
Strike:           $694
Vencimiento:      20260114
Prima total:      $550.00 (1 contrato)
Break-even sup:   $699.50 (+0.79%)
Break-even inf:   $688.50 (-0.79%)
================================================================================

Confirmar envío? [y/N]: y
Enviando orden...
Order ID: 12345
Estado: Submitted
```

In [6]:
# ==============================================================================
# CELDA 5A: FUNCIONES DE ENVÍO DE ÓRDENES
# ==============================================================================
# Estas funciones reutilizan la conexión `ib` y los datos ya calculados.
# NO reconectan ni buscan contratos desde cero.

from ib_insync import Option, Contract, ComboLeg, MarketOrder, LimitOrder

def create_option_contract_from_chain(
    df_chain: pd.DataFrame,
    strike: float,
    expiry: str,
    right: str,
    symbol: str = 'SPY'
) -> Option:
    """
    Crea un contrato de opción usando datos de df_option_chain.
    Evita llamadas adicionales a qualifyContractsAsync al usar conIds existentes.
    
    Parámetros:
    - df_chain: DataFrame con la cadena de opciones (debe tener call_conId, put_conId)
    - strike: Precio de ejercicio
    - expiry: Fecha de vencimiento (YYYYMMDD)
    - right: 'C' para Call, 'P' para Put
    - symbol: Símbolo del subyacente (default: 'SPY')
    
    Returns:
    - Option: Contrato de opción con conId asignado
    """
    row = df_chain[df_chain['strike'] == strike]
    if row.empty:
        available = df_chain['strike'].tolist()
        raise ValueError(f"Strike {strike} no encontrado. Disponibles: {available[:10]}...")
    
    row = row.iloc[0]
    
    if right.upper() == 'C':
        con_id = int(row['call_conId'])
    else:
        con_id = int(row['put_conId'])
    
    contract = Option(
        symbol=symbol,
        lastTradeDateOrContractMonth=expiry,
        strike=strike,
        right=right,
        exchange='SMART',
        currency='USD',
        tradingClass=symbol
    )
    contract.conId = con_id
    
    return contract


def create_straddle_combo_contract(
    call_conId: int,
    put_conId: int,
    symbol: str = 'SPY',
    action: str = 'BUY'
) -> Contract:
    """
    Crea un contrato BAG (combo) para un straddle.
    
    Parámetros:
    - call_conId: ID del contrato CALL
    - put_conId: ID del contrato PUT
    - symbol: Símbolo del subyacente
    - action: 'BUY' para long straddle, 'SELL' para short straddle
    
    Returns:
    - Contract: Contrato BAG con ambas patas
    """
    bag = Contract()
    bag.symbol = symbol
    bag.secType = 'BAG'
    bag.currency = 'USD'
    bag.exchange = 'SMART'
    
    # Pata CALL
    leg_call = ComboLeg()
    leg_call.conId = call_conId
    leg_call.ratio = 1
    leg_call.action = action
    leg_call.exchange = 'SMART'
    
    # Pata PUT
    leg_put = ComboLeg()
    leg_put.conId = put_conId
    leg_put.ratio = 1
    leg_put.action = action
    leg_put.exchange = 'SMART'
    
    bag.comboLegs = [leg_call, leg_put]
    
    return bag


def display_order_summary(
    strike: float,
    expiry: str,
    call_conId: int,
    put_conId: int,
    call_price: float,
    put_price: float,
    order_type: str,
    mode: str,
    quantity: int = 1
) -> None:
    """
    Muestra un resumen detallado de la orden antes de enviarla.
    
    Incluye: detalles del contrato, precios, costo total, break-evens.
    """
    total_cost = call_price + put_price
    total_premium = total_cost * 100 * quantity
    be_up = strike + total_cost
    be_down = strike - total_cost
    
    print("\n" + "="*80)
    print("RESUMEN DE ORDEN - LONG STRADDLE")
    print("="*80)
    
    print(f"\n{'CONTRATO':<20} {'VALOR':<30}")
    print("-"*50)
    print(f"{'Símbolo:':<20} SPY")
    print(f"{'Strike:':<20} ${strike:.0f}")
    print(f"{'Vencimiento:':<20} {expiry}")
    print(f"{'CALL conId:':<20} {call_conId}")
    print(f"{'PUT conId:':<20} {put_conId}")
    
    print(f"\n{'PRECIOS':<20} {'VALOR':<30}")
    print("-"*50)
    print(f"{'CALL (ask):':<20} ${call_price:.2f}")
    print(f"{'PUT (ask):':<20} ${put_price:.2f}")
    print(f"{'Total unitario:':<20} ${total_cost:.2f}")
    print(f"{'Prima total:':<20} ${total_premium:.2f} ({quantity} contrato(s))")
    
    print(f"\n{'BREAK-EVEN':<20} {'VALOR':<30}")
    print("-"*50)
    print(f"{'BE Superior:':<20} ${be_up:.2f} (+{(be_up/strike-1)*100:.2f}%)")
    print(f"{'BE Inferior:':<20} ${be_down:.2f} ({(be_down/strike-1)*100:.2f}%)")
    
    print(f"\n{'EJECUCIÓN':<20} {'VALOR':<30}")
    print("-"*50)
    print(f"{'Modo:':<20} {mode}")
    print(f"{'Tipo orden:':<20} {order_type}")
    print(f"{'Cantidad:':<20} {quantity}")
    
    print("="*80)


async def send_order_with_confirmation(
    ib_conn,
    contract: Contract,
    order,
    require_confirmation: bool = True,
    timeout_seconds: int = 30
):
    """
    Envía una orden con confirmación interactiva y monitoreo de estado.
    
    Parámetros:
    - ib_conn: Conexión IB activa
    - contract: Contrato a operar
    - order: Orden a enviar (MarketOrder o LimitOrder)
    - require_confirmation: Si True, pide confirmación antes de enviar
    - timeout_seconds: Tiempo máximo de espera para fill
    
    Returns:
    - Trade: Objeto trade con información de la orden
    """
    if not ib_conn.isConnected():
        raise RuntimeError("No hay conexión activa con IB. Ejecuta la celda de conexión primero.")
    
    print(f"\nContrato: {contract}")
    print(f"Orden: {order}")
    
    if require_confirmation:
        confirm = input("\nConfirmar envío? [y/N]: ").strip().lower()
        if confirm != 'y':
            print("Orden cancelada por el usuario.")
            return None
    
    print("\nEnviando orden...")
    trade = ib_conn.placeOrder(contract, order)
    
    print(f"Order ID: {trade.order.orderId}")
    print(f"Estado inicial: {trade.orderStatus.status}")
    
    # Monitorear estado
    elapsed = 0
    while not trade.isDone() and elapsed < timeout_seconds:
        await asyncio.sleep(1)
        elapsed += 1
        if elapsed % 5 == 0:
            print(f"  [{elapsed}s] Estado: {trade.orderStatus.status}")
    
    final_status = trade.orderStatus.status
    print(f"\nEstado final: {final_status}")
    
    if final_status == 'Filled':
        print(f"Precio de ejecución: ${trade.orderStatus.avgFillPrice:.2f}")
        print(f"Cantidad ejecutada: {trade.orderStatus.filled}")
    elif final_status in ['Submitted', 'PreSubmitted', 'PendingSubmit']:
        print("La orden está pendiente. Verifica en TWS/IB Gateway.")
    elif final_status == 'Inactive':
        print("Orden inactiva. Posible causa: mercado cerrado o restricción.")
    
    return trade


print("="*80)
print("FUNCIONES DE ENVÍO DE ÓRDENES CARGADAS")
print("="*80)
print("\nFunciones disponibles:")
print("  - create_option_contract_from_chain(): Crea contrato usando df_option_chain")
print("  - create_straddle_combo_contract(): Crea contrato BAG para straddle")
print("  - display_order_summary(): Muestra resumen de la orden")
print("  - send_order_with_confirmation(): Envía orden con confirmación")
print("\nEjecuta la siguiente celda para enviar órdenes.")

FUNCIONES DE ENVÍO DE ÓRDENES CARGADAS

Funciones disponibles:
  - create_option_contract_from_chain(): Crea contrato usando df_option_chain
  - create_straddle_combo_contract(): Crea contrato BAG para straddle
  - display_order_summary(): Muestra resumen de la orden
  - send_order_with_confirmation(): Envía orden con confirmación

Ejecuta la siguiente celda para enviar órdenes.


In [7]:
# ==============================================================================
# CELDA 5B: EJECUCIÓN DE ÓRDENES
# ==============================================================================
# Usa las funciones de la celda anterior y las variables calculadas previamente.
# Permite elegir entre modo COMBO o INDIVIDUAL, y tipo de orden MKT o LMT.

# -----------------------------------------------------------------------------
# CONFIGURACIÓN (MODIFICAR SEGÚN NECESIDAD)
# -----------------------------------------------------------------------------
MODO = "COMBO"          # "COMBO" = Una orden BAG | "INDIVIDUAL" = Órdenes separadas
ORDER_TYPE = "LMT"      # "MKT" = Market Order | "LMT" = Limit Order
QUANTITY = 1            # Número de contratos a operar
REQUIRE_CONFIRM = True  # Si True, pide confirmación antes de enviar

# -----------------------------------------------------------------------------
# VERIFICACIONES INICIALES
# -----------------------------------------------------------------------------
print("="*80)
print("ENVÍO DE ÓRDENES - LONG STRADDLE SPY")
print("="*80 + "\n")

# Verificar conexión
if not ib.isConnected():
    raise RuntimeError(
        "No hay conexión activa con IB.\n"
        "Ejecuta la celda de conexión (apartado 1) primero."
    )
print(f"✓ Conexión activa (clientId={ib.client.clientId})")

# Verificar que existen las variables necesarias
required_vars = ['df_with_prices', 'df_option_chain', 'expiry_date', 'spy_price']
for var in required_vars:
    if var not in dir():
        raise RuntimeError(f"Variable '{var}' no encontrada. Ejecuta las celdas anteriores.")
print(f"✓ Variables requeridas disponibles")

# -----------------------------------------------------------------------------
# 1. SELECCIONAR STRIKE ATM
# -----------------------------------------------------------------------------
print(f"\n1. Seleccionando strike ATM...")

# Calcular distancia al precio actual si no existe
if 'distance' not in df_with_prices.columns:
    df_with_prices['distance'] = abs(df_with_prices['strike'] - spy_price)

atm_row = df_with_prices.loc[df_with_prices['distance'].idxmin()]
selected_strike = atm_row['strike']
call_conId = int(atm_row['call_conId'])
put_conId = int(atm_row['put_conId'])

print(f"   Strike ATM: ${selected_strike:.0f} (más cercano a SPY=${spy_price:.2f})")
print(f"   CALL conId: {call_conId}")
print(f"   PUT conId:  {put_conId}")

# -----------------------------------------------------------------------------
# 2. OBTENER PRECIOS DE MERCADO
# -----------------------------------------------------------------------------
print(f"\n2. Precios de mercado:")

# Usar ask para compra (worst case), con fallback a last
call_bid = atm_row['call_bid'] if pd.notna(atm_row['call_bid']) else 0
call_ask = atm_row['call_ask'] if pd.notna(atm_row['call_ask']) else atm_row['call_last']
put_bid = atm_row['put_bid'] if pd.notna(atm_row['put_bid']) else 0
put_ask = atm_row['put_ask'] if pd.notna(atm_row['put_ask']) else atm_row['put_last']

# Asegurar que tenemos precios válidos
if pd.isna(call_ask) or pd.isna(put_ask):
    raise RuntimeError("No hay precios válidos para el strike ATM. Verifica df_with_prices.")

call_ask = float(call_ask)
put_ask = float(put_ask)

print(f"   CALL: bid=${call_bid:.2f} / ask=${call_ask:.2f}")
print(f"   PUT:  bid=${put_bid:.2f} / ask=${put_ask:.2f}")
print(f"   Costo total estimado: ${call_ask + put_ask:.2f} x 100 = ${(call_ask + put_ask) * 100:.2f}")

# -----------------------------------------------------------------------------
# 3. MOSTRAR CONFIGURACIÓN
# -----------------------------------------------------------------------------
print(f"\n3. Configuración de ejecución:")
print(f"   Modo:        {MODO} {'(orden única BAG)' if MODO == 'COMBO' else '(órdenes separadas)'}")
print(f"   Tipo orden:  {ORDER_TYPE} {'(Market - ejecución inmediata)' if ORDER_TYPE == 'MKT' else '(Limit - precio controlado)'}")
print(f"   Cantidad:    {QUANTITY} contrato(s)")
print(f"   Confirmación: {'Sí' if REQUIRE_CONFIRM else 'No'}")

# -----------------------------------------------------------------------------
# 4. CREAR Y ENVIAR ÓRDENES
# -----------------------------------------------------------------------------
print(f"\n4. Creando órdenes...")

if MODO == "COMBO":
    # =========================================================================
    # MODO COMBO: Una sola orden BAG para ambas patas
    # =========================================================================
    print("   Modo COMBO: Creando contrato BAG...")
    
    combo_contract = create_straddle_combo_contract(call_conId, put_conId, 'SPY', 'BUY')
    
    if ORDER_TYPE == "MKT":
        combo_order = MarketOrder('BUY', QUANTITY)
    else:
        # Para LMT en combo, el precio es la suma de ambas primas
        limit_price = round(call_ask + put_ask, 2)
        combo_order = LimitOrder('BUY', QUANTITY, limit_price)
        print(f"   Precio límite COMBO: ${limit_price:.2f}")
    
    # Mostrar resumen
    display_order_summary(
        strike=selected_strike,
        expiry=expiry_date,
        call_conId=call_conId,
        put_conId=put_conId,
        call_price=call_ask,
        put_price=put_ask,
        order_type=ORDER_TYPE,
        mode=MODO,
        quantity=QUANTITY
    )
    
    # Enviar orden
    print("\n--- ENVIANDO ORDEN COMBO ---")
    combo_trade = await send_order_with_confirmation(
        ib, combo_contract, combo_order, 
        require_confirmation=REQUIRE_CONFIRM
    )

else:
    # =========================================================================
    # MODO INDIVIDUAL: Órdenes separadas para CALL y PUT
    # =========================================================================
    print("   Modo INDIVIDUAL: Creando contratos separados...")
    
    # Crear contratos individuales usando df_option_chain
    call_contract = create_option_contract_from_chain(df_option_chain, selected_strike, expiry_date, 'C')
    put_contract = create_option_contract_from_chain(df_option_chain, selected_strike, expiry_date, 'P')
    
    if ORDER_TYPE == "MKT":
        call_order = MarketOrder('BUY', QUANTITY)
        put_order = MarketOrder('BUY', QUANTITY)
    else:
        call_order = LimitOrder('BUY', QUANTITY, round(call_ask, 2))
        put_order = LimitOrder('BUY', QUANTITY, round(put_ask, 2))
        print(f"   Precio límite CALL: ${call_ask:.2f}")
        print(f"   Precio límite PUT:  ${put_ask:.2f}")
    
    # Mostrar resumen
    display_order_summary(
        strike=selected_strike,
        expiry=expiry_date,
        call_conId=call_conId,
        put_conId=put_conId,
        call_price=call_ask,
        put_price=put_ask,
        order_type=ORDER_TYPE,
        mode=MODO,
        quantity=QUANTITY
    )
    
    # Enviar órdenes
    print("\n--- ENVIANDO ORDEN CALL ---")
    call_trade = await send_order_with_confirmation(
        ib, call_contract, call_order,
        require_confirmation=REQUIRE_CONFIRM
    )
    
    print("\n--- ENVIANDO ORDEN PUT ---")
    put_trade = await send_order_with_confirmation(
        ib, put_contract, put_order,
        require_confirmation=REQUIRE_CONFIRM
    )

# -----------------------------------------------------------------------------
# 5. RESUMEN FINAL
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("PROCESO COMPLETADO")
print("="*80)
print(f"\nRevisa el estado de las órdenes en TWS/IB Gateway.")
print(f"Si el mercado está cerrado, las órdenes quedarán pendientes.")

ENVÍO DE ÓRDENES - LONG STRADDLE SPY

✓ Conexión activa (clientId=352)
✓ Variables requeridas disponibles

1. Seleccionando strike ATM...
   Strike ATM: $694 (más cercano a SPY=$693.96)
   CALL conId: 837821900
   PUT conId:  837823406

2. Precios de mercado:
   CALL: bid=$3.49 / ask=$3.50
   PUT:  bid=$2.72 / ask=$2.74
   Costo total estimado: $6.24 x 100 = $624.00

3. Configuración de ejecución:
   Modo:        COMBO (orden única BAG)
   Tipo orden:  LMT (Limit - precio controlado)
   Cantidad:    1 contrato(s)
   Confirmación: Sí

4. Creando órdenes...
   Modo COMBO: Creando contrato BAG...
   Precio límite COMBO: $6.24

RESUMEN DE ORDEN - LONG STRADDLE

CONTRATO             VALOR                         
--------------------------------------------------
Símbolo:             SPY
Strike:              $694
Vencimiento:         20260116
CALL conId:          837821900
PUT conId:           837823406

PRECIOS              VALOR                         
-----------------------------------

---

## 6. Recuperar Posiciones y Cartera Actual

### Objetivo
Obtener un reporte detallado de las posiciones abiertas, incluyendo P&L (Ganancias/Pérdidas) y valor de mercado.

### Diferencia entre `positions()` y `portfolio()`
- **`positions()`**: Solo devuelve el inventario (qué tienes, cuánto tienes y a qué precio compraste).
- **`portfolio()`**: Devuelve el inventario + **datos** de mercado (precio actual, valor de mercado, P&L no realizado).

### ¿Qué hace esta celda?
1. Solicita la actualización de la cuenta (`reqAccountUpdates`).
2. Descarga los ítems del portafolio.
3. Construye un `DataFrame` formateado, distinguiendo entre Acciones (`STK`) y Opciones (`OPT`).
4. Calcula totales de P&L de la cartera.

In [8]:
import pandas as pd
import asyncio

async def get_portfolio_robust():
    print("="*80)
    print("CARTERA DETALLADA (MODO ROBUSTO)")
    print("="*80)

    # 1. Verificar conexión
    if not ib.isConnected():
        print("⚠ No hay conexión. Intentando reconectar...")
        try:
            await ib.connectAsync('127.0.0.1', 7497, clientId=random.randint(1000, 9999))
        except Exception as e:
            print(f"✗ Error crítico de conexión: {e}")
            return pd.DataFrame()

    print("1. Leyendo inventario...")
    
    # 2. Intentar leer la caché local primero (NO BLOQUEA)
    portfolio_items = ib.portfolio()

    # 3. Solo si está vacío, pedimos actualización con TIMEOUT DE SEGURIDAD
    if not portfolio_items:
        print("   (Caché vacía, solicitando actualización segura...)")
        ib.reqAccountUpdates(True)
        try:
            # Esperamos máximo 4 segundos a que llegue algo
            await asyncio.wait_for(asyncio.sleep(4), timeout=5.0)
            portfolio_items = ib.portfolio()
        except asyncio.TimeoutError:
            print("⚠ Tiempo de espera agotado (Timeout). Mostrando lo que se pudo recuperar.")
        except Exception as e:
            print(f"⚠ Error en actualización: {e}")

    print(f"✓ {len(portfolio_items)} posiciones individuales encontradas.\n")

    # 4. Construir DataFrame con detalle completo
    data = []
    for item in portfolio_items:
        contract = item.contract
        
        # Construir descripción detallada para comparar con TWS
        if contract.secType == 'OPT':
            # Formato: SPY 20260114 694.0 CALL
            right_full = "CALL" if contract.right == 'C' else "PUT"
            desc = f"{contract.symbol} {contract.lastTradeDateOrContractMonth} {contract.strike} {right_full}"
            security_type = "Opción"
        else:
            desc = contract.symbol
            security_type = "Acción"

        data.append({
            'Tipo': security_type,
            'Descripción': desc,
            'Posición': item.position,
            'Precio Mercado': item.marketPrice,
            'Coste Medio': item.averageCost,
            'Valor Mercado': item.marketValue,
            'P&L No Realizado': item.unrealizedPNL,
            'P&L Realizado': item.realizedPNL
        })

    df = pd.DataFrame(data)

    # 5. Formateo y visualización
    if not df.empty:
        # Ordenar: Primero acciones, luego opciones por descripción
        df = df.sort_values(by=['Tipo', 'Descripción'], ascending=[True, True])
        
        # Calcular totales para verificar coherencia con TWS
        total_pnl = df['P&L No Realizado'].sum()
        total_val = df['Valor Mercado'].sum()
        
        print(f"Total P&L No Realizado: ${total_pnl:,.2f}")
        print(f"Total Valor de Mercado: ${total_val:,.2f}")
        print("-" * 80)
        
    return df

# Ejecución segura
try:
    df_portfolio = await get_portfolio_robust()
    display(df_portfolio)
except Exception as e:
    print(f"Error inesperado: {e}")

CARTERA DETALLADA (MODO ROBUSTO)
1. Leyendo inventario...
✓ 4 posiciones individuales encontradas.

Total P&L No Realizado: $-390.27
Total Valor de Mercado: $1,977.00
--------------------------------------------------------------------------------


,Tipo,Descripción,Posición,Precio Mercado,Coste Medio,Valor Mercado,P&L No Realizado,P&L Realizado
0,Opción,SPY 20260114 694.0 CALL,2.0,2.02,233.338750,404.0,-62.68,0.0
1,Opción,SPY 20260114 694.0 PUT,1.0,1.54,263.628800,154.0,-109.63,0.0
2,Opción,SPY 20260115 695.0 CALL,3.0,2.14,315.825417,642.0,-305.48,0.0
3,Opción,SPY 20260115 695.0 PUT,3.0,2.59,229.825417,777.0,87.52,0.0


---

## [LAST] Desconectar de TWS/IB Gateway

### Objetivo
Cerrar la conexión con IBKR de forma limpia y liberar el CLIENT_ID.

### ¿Qué hace esta celda?

#### **Función `disconnect_from_ib_async()`**

1. **Verifica si hay conexión activa**
   - Comprueba `ib.isConnected()`
   - Si no hay conexión, muestra advertencia y termina

2. **Cancela suscripciones activas**
   - Cancela todas las suscripciones de market data
   - Limpia recursos antes de desconectar
   - Previene warnings de IBKR

3. **Desconecta limpiamente**
   - Ejecuta `ib.disconnect()`
   - Espera 1 segundo para asegurar desconexión completa
   - Libera el CLIENT_ID para futuros usos

4. **Confirma desconexión**
   - Muestra CLIENT_ID liberado
   - Confirma que la sesión finalizó correctamente

### Resultado esperado



In [9]:
async def disconnect_from_ib_async():
    """Desconecta de TWS/IB Gateway de forma segura"""
    
    print("="*80)
    print("CERRANDO CONEXIÓN")
    print("="*80 + "\n")
    
    try:
        if ib.isConnected():
            client_id = ib.client.clientId
            print(f"Desconectando clientId={client_id}...")
            
            # Cancelar todas las suscripciones de market data activas
            try:
                ib.cancelMktData()
            except:
                pass
            
            # Desconectar
            ib.disconnect()
            await asyncio.sleep(1)
            
            print("✓ Desconectado exitosamente de TWS/IB Gateway")
            print(f"  ClientId {client_id} liberado\n")
        else:
            print("⚠ No hay conexión activa para cerrar\n")
            
    except Exception as e:
        print(f"✗ Error al desconectar: {e}")
        print("  (La conexión puede haberse cerrado ya)\n")

# Ejecutar desconexión
await disconnect_from_ib_async()
print("✓ Sesión finalizada")

CERRANDO CONEXIÓN

Desconectando clientId=352...
⚠ Desconectado de TWS/IB Gateway
✓ Desconectado exitosamente de TWS/IB Gateway
  ClientId 352 liberado

✓ Sesión finalizada
